# PHẦN 2: TẦNG TRUY XUẤT ỨNG VIÊN

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'polars', 'faiss-cpu'])
import os
import pandas as pd
import numpy as np
import polars as pl
import torch
import torch.nn as nn
import scipy.sparse as sp
import faiss
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
DRIVE_DATA_DIR = '/content/drive/MyDrive/DM_test'
LOCAL_DATA_DIR = '/content/'

TRAIN_FILE = 'train_interactions.parquet'
TRAIN_PATH_DRIVE = os.path.join(DRIVE_DATA_DIR, TRAIN_FILE)
TRAIN_PATH = os.path.join(LOCAL_DATA_DIR, TRAIN_FILE)

if not os.path.exists(TRAIN_PATH):
    print(f"Copying {TRAIN_FILE} from Drive to local...")
    shutil.copy(TRAIN_PATH_DRIVE, TRAIN_PATH)

CAND_PATH = os.path.join(LOCAL_DATA_DIR, 'candidates_phase2.parquet')
CAND_PATH_DRIVE = os.path.join(DRIVE_DATA_DIR, 'candidates_phase2.parquet')
MAX_LEN   = 50
EMBED_DIM = 64

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def load_data(path):
    df = pl.read_parquet(
        path,
        columns=['mapped_user_id', 'mapped_item_id', 'rating', 'timestamp']
    ).to_pandas()
    return df

In [ ]:
import gc
import polars as pl

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import faiss
from tqdm.auto import tqdm

# 1. SASRec Data Prep with Polars (RAM Efficient)
print("Đang tiền xử lý chuỗi bằng Polars...")
df_train_pl = pl.read_parquet(TRAIN_PATH, columns=['mapped_user_id', 'mapped_item_id', 'timestamp'])

num_users = df_train_pl['mapped_user_id'].max() + 1
num_items = df_train_pl['mapped_item_id'].max() + 1

# Group sequences in Polars
user_seqs_pl = (
    df_train_pl.sort(['mapped_user_id', 'timestamp'])
    .group_by('mapped_user_id')
    .agg(pl.col('mapped_item_id'))
)

mapped_user_ids = user_seqs_pl['mapped_user_id'].to_numpy()
item_lists = user_seqs_pl['mapped_item_id'].to_list()

X_sas_train = np.zeros((len(item_lists), MAX_LEN), dtype=np.int32)
for idx, seq in enumerate(item_lists):
    s = seq[-MAX_LEN:]
    X_sas_train[idx, MAX_LEN-len(s):] = s

del df_train_pl, user_seqs_pl, item_lists
gc.collect()

class SASRec(nn.Module):
    def __init__(self, n_items, embed_dim, max_len):
        super().__init__()
        self.item_emb = nn.Embedding(n_items, embed_dim, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len, embed_dim)
        layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=1, batch_first=True, dim_feedforward=embed_dim*2)
        self.transformer = nn.TransformerEncoder(layer, num_layers=1)
    def forward(self, seqs):
        pos = torch.arange(seqs.size(1), device=seqs.device).unsqueeze(0).expand_as(seqs)
        mask = (seqs == 0)
        out = self.transformer(self.item_emb(seqs) + self.pos_emb(pos), src_key_padding_mask=mask)
        return out[:, -1, :]

model_sasrec = SASRec(num_items, EMBED_DIM, MAX_LEN).to(device)
optimizer = torch.optim.Adam(model_sasrec.parameters(), lr=0.005)
criterion = nn.CrossEntropyLoss(ignore_index=0)

model_sasrec.train()
epochs = 30 # Reduced epochs for faster execution in Colab
batch_size = 128

for ep in range(epochs):
    idx = np.random.permutation(len(X_sas_train))
    loss_ep, t_batches = 0, 0
    pbar = tqdm(range(0, len(X_sas_train), batch_size), desc=f"Epoch {ep+1}/{epochs}", leave=False)
    for i in pbar:
        b_idx = idx[i:i+batch_size]
        batch_seqs = torch.tensor(X_sas_train[b_idx], dtype=torch.long).to(device)
        u_reps = model_sasrec(batch_seqs[:, :-1])
        logits = torch.matmul(u_reps, model_sasrec.item_emb.weight.T)
        loss = criterion(logits, batch_seqs[:, -1])
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        loss_ep += loss.item(); t_batches += 1
        pbar.set_postfix(loss=loss_ep/t_batches)
    if (ep+1) % 5 == 0: print(f"SASRec Epoch {ep+1} Average Loss: {loss_ep/t_batches:.4f}")

model_sasrec.eval()
with torch.no_grad():
    i_embs = torch.nn.functional.normalize(model_sasrec.item_emb.weight[1:], p=2, dim=1).cpu().numpy().astype('float32')
    index_sas = faiss.IndexFlatIP(EMBED_DIM)
    index_sas.add(i_embs)
    all_top_idx = []
    X_tensor = torch.tensor(X_sas_train, dtype=torch.long)
    for i in tqdm(range(0, len(X_tensor), 2000), desc="Inference"):
        u_reps = model_sasrec(X_tensor[i:i+2000].to(device))
        u_reps = torch.nn.functional.normalize(u_reps, p=2, dim=1).cpu().numpy().astype('float32')
        _, idx = index_sas.search(u_reps, 100)
        all_top_idx.append(idx.astype('int32'))

df_sasrec = pd.DataFrame({
    'mapped_user_id': np.repeat(mapped_user_ids, 100),
    'mapped_item_id': (np.vstack(all_top_idx).flatten() + 1).astype('int32'),
    'sasrec_rank': np.tile(np.arange(1, 101), len(mapped_user_ids)).astype('float32')
})
del X_sas_train, X_tensor, all_top_idx; gc.collect()

Đang tiền xử lý chuỗi bằng Polars...


/tmp/ipykernel_17997/887560088.py:42: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  self.transformer = nn.TransformerEncoder(layer, num_layers=1)


Epoch 1/30:   0%|          | 0/70851 [00:00<?, ?it/s]

Epoch 2/30:   0%|          | 0/70851 [00:00<?, ?it/s]

In [ ]:
import scipy.sparse as sp
# 2. LightGCN
print("Đang chuẩn bị dữ liệu LightGCN...")
df_train_pd = load_data(TRAIN_PATH)
u_idx, i_idx = df_train_pd['mapped_user_id'].to_numpy(), df_train_pd['mapped_item_id'].to_numpy()
adj = sp.coo_matrix((np.ones(len(u_idx)), (u_idx, i_idx + num_users)), shape=(num_users+num_items, num_users+num_items))
adj = adj + adj.T

d_inv = np.power(np.array(adj.sum(1)), -0.5).flatten()
d_inv[np.isinf(d_inv)] = 0.
d_mat = sp.diags(d_inv)
norm_adj = d_mat.dot(adj).dot(d_mat).tocoo()
indices = torch.LongTensor(np.vstack((norm_adj.row, norm_adj.col)))
norm_adj_t = torch.sparse_coo_tensor(indices, torch.FloatTensor(norm_adj.data), norm_adj.shape).to(device)

class LightGCN(nn.Module):
    def __init__(self, u, i, dim):
        super().__init__()
        self.u_emb = nn.Embedding(u, dim); self.i_emb = nn.Embedding(i, dim)
        nn.init.normal_(self.u_emb.weight, std=0.1); nn.init.normal_(self.i_emb.weight, std=0.1)
    def forward(self, adj):
        emb = torch.cat([self.u_emb.weight, self.i_emb.weight])
        all_embs = [emb]
        for _ in range(2): emb = torch.sparse.mm(adj, emb); all_embs.append(emb)
        return torch.split(torch.stack(all_embs, dim=1).mean(dim=1), [num_users, num_items])

model_lgcn = LightGCN(num_users, num_items, EMBED_DIM).to(device)
optimizer = torch.optim.Adam(model_lgcn.parameters(), lr=0.001)
pos_pairs = df_train_pd[['mapped_user_id', 'mapped_item_id']].to_numpy()

for ep in range(50):
    np.random.shuffle(pos_pairs)
    loss_ep, t_batches = 0, 0
    for i in range(0, len(pos_pairs), 10240):
        batch = pos_pairs[i:i+10240]
        u_reps, i_reps = model_lgcn(norm_adj_t)
        pos_scores = (u_reps[batch[:,0]] * i_reps[batch[:,1]]).sum(1)
        neg_items = np.random.randint(0, num_items, size=len(batch))
        neg_scores = (u_reps[batch[:,0]] * i_reps[neg_items]).sum(1)
        loss = -torch.log(torch.sigmoid(pos_scores - neg_scores) + 1e-8).mean()
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        loss_ep += loss.item(); t_batches += 1
    if (ep+1)%10==0: print(f"LightGCN Epoch {ep+1} Loss: {loss_ep/t_batches:.4f}")

model_lgcn.eval()
with torch.no_grad():
    u_e, i_e = model_lgcn(norm_adj_t)
    u_e = torch.nn.functional.normalize(u_e, p=2, dim=1).cpu().numpy().astype('float32')
    i_e = torch.nn.functional.normalize(i_e, p=2, dim=1).cpu().numpy().astype('float32')

index = faiss.IndexFlatIP(EMBED_DIM)
index.add(i_e)
_, idx_lgcn = index.search(u_e, 100)

df_lightgcn = pd.DataFrame({
    'mapped_user_id': np.repeat(np.arange(num_users), 100),
    'mapped_item_id': idx_lgcn.flatten().astype('int32'),
    'lightgcn_rank': np.tile(np.arange(1, 101), num_users).astype('float32')
})
del df_train_pd, pos_pairs, u_e, i_e, idx_lgcn; gc.collect()

In [ ]:
import gc
import polars as pl

# Ensure DataFrames exist
if 'df_sasrec' in locals() and 'df_lightgcn' in locals():
    print("Đang chuyển đổi và gộp bảng bằng Polars...")
    pl_sasrec = pl.from_pandas(df_sasrec)
    pl_lightgcn = pl.from_pandas(df_lightgcn)

    df_union = pl_sasrec.join(pl_lightgcn, on=['mapped_user_id', 'mapped_item_id'], how='full', coalesce=True)

    del df_sasrec, df_lightgcn, pl_sasrec, pl_lightgcn
    gc.collect()

    print(f"Số lượng ứng viên sau khi gộp: {len(df_union)}")
    df_union.write_parquet(CAND_PATH)
    print("Copying candidates back to Drive...")
    shutil.copy(CAND_PATH, CAND_PATH_DRIVE)
    print("Đã lưu Candidates thành công.")
else:
    print("Lỗi: Không tìm thấy df_sasrec hoặc df_lightgcn. Vui lòng chạy các ô training trước.")